In [1]:
import os
import numpy as np
import h5py
from math import ceil

In [2]:
root_dir = "/system/user/publicdata/gyrokinetics"
folder = "cyclone4_2"
dir_in = os.path.join(root_dir, folder)
dir_out = "/system/user/publicdata/gyrokinetics_preprocessed"
if not os.path.exists(dir_out):
    os.makedirs(dir_out)

In [25]:
def K_files(directory):
    files = os.listdir(directory)
    digit_files = sorted(
        [file for file in files if file.isdigit()], key=lambda x: int(x)
    )
    k_files = sorted(
        [file for file in files if file.startswith("K") and not file.endswith(".dat")]
    )
    return k_files + digit_files


def poten_files(directory):
    files = os.listdir(directory)
    poten_files = sorted([file for file in files if file.startswith("Poten")])
    timestep_slices = [int(f.replace("Poten", "")) for f in poten_files]
    return poten_files, np.array(timestep_slices) - 1

In [26]:
ks = K_files(dir_in)
potens, ts_slices = poten_files(dir_in)

In [27]:
# get timestamps
ts = []
for k in ks:
    # load corresponding timestep
    with open(os.path.join(dir_in, k + ".dat"), "r") as file:
        for line in file:
            line_split = line.split("=")
            if line_split[0].strip() == "TIME":
                time = float(line_split[1].strip().strip(",").strip())
                ts.append(time)
timesteps = np.array(ts)

# read helper vars
time = np.loadtxt(os.path.join(dir_in, "time.dat"))
sgrid = np.loadtxt(os.path.join(dir_in, "sgrid"))
xphi = np.loadtxt(os.path.join(dir_in, "xphi"))
yphi = np.loadtxt(os.path.join(dir_in, "yphi"))
fluxes = np.loadtxt(os.path.join(dir_in, "fluxes.dat"))
kyspec = np.loadtxt(os.path.join(dir_in, "kyspec"))
krho = np.loadtxt(os.path.join(dir_in, "krho"))
vpgr = np.loadtxt(os.path.join(dir_in, "vpgr.dat"))
vperp = np.loadtxt(os.path.join(dir_in, "vperp.dat"))
# number of parallel direction grid points
ns = sgrid.shape[1] if len(sgrid.shape) > 1 else sgrid.shape[0]
# number of x, y grid points (in real space)
nx, ny = xphi.shape[1], xphi.shape[0]
# number of modes in x and y direction
nkx, nky = krho.shape[1], krho.shape[0]
# get velocity space resolutions
nvpar, nmu = vpgr.shape[1], vpgr.shape[0]

# load fluxes
fluxes = np.loadtxt(os.path.join(dir_in, "fluxes.dat"))[:, 1]
fluxes = fluxes[ts_slices]

# create h5 file with timestamps and field data
h5_filename = os.path.join(dir_out, folder + ".h5")
with h5py.File(h5_filename, "w") as file:
    # group for metadata (e.g. timesteps)
    metadata_group = file.create_group("metadata")
    metadata_group.create_dataset("timesteps", data=timesteps)
    metadata_group.create_dataset("fluxes", data=fluxes)

    # group for our 6D field data
    data_group = file.create_group("data")
    for idx, (k, pot) in enumerate(zip(ks, potens)):

        # Load the full distribution function data
        with open(os.path.join(dir_in, k), "rb") as fid:
            ff = np.fromfile(fid, dtype=np.float64)

        # Reshape the distribution function
        knth = np.reshape(ff, (2, nvpar, nmu, ns, nkx, nky), order="F").astype(
            "float32"
        )

        # Add the reshaped data as a dataset to the "data" group
        k_name = "timestep_" + str(idx).zfill(2)
        data_group.create_dataset(k_name, data=knth)

        # load the potential field
        a = np.loadtxt(os.path.join(dir_in, pot))
        phi = np.reshape(a, (nx, ns, ny)).astype("float32")
        poten_name = "poten_" + str(idx).zfill(2)
        data_group.create_dataset(poten_name, data=phi)

        print(f"Added dataset '{k_name}'+'{poten_name}' to HDF5 file.")

Added dataset 'timestep_00'+'poten_00' to HDF5 file.
Added dataset 'timestep_01'+'poten_01' to HDF5 file.
Added dataset 'timestep_02'+'poten_02' to HDF5 file.
Added dataset 'timestep_03'+'poten_03' to HDF5 file.
Added dataset 'timestep_04'+'poten_04' to HDF5 file.
Added dataset 'timestep_05'+'poten_05' to HDF5 file.
Added dataset 'timestep_06'+'poten_06' to HDF5 file.
Added dataset 'timestep_07'+'poten_07' to HDF5 file.
Added dataset 'timestep_08'+'poten_08' to HDF5 file.
Added dataset 'timestep_09'+'poten_09' to HDF5 file.
Added dataset 'timestep_10'+'poten_10' to HDF5 file.
Added dataset 'timestep_11'+'poten_11' to HDF5 file.
Added dataset 'timestep_12'+'poten_12' to HDF5 file.
Added dataset 'timestep_13'+'poten_13' to HDF5 file.
Added dataset 'timestep_14'+'poten_14' to HDF5 file.
Added dataset 'timestep_15'+'poten_15' to HDF5 file.
Added dataset 'timestep_16'+'poten_16' to HDF5 file.
Added dataset 'timestep_17'+'poten_17' to HDF5 file.
Added dataset 'timestep_18'+'poten_18' to HDF5

In [28]:
# read in the structure and example field of the created h5 file
with h5py.File(h5_filename, "r") as f:
    # Read the 'metadata/timesteps' dataset
    timesteps = f["metadata/timesteps"][:]
    print("Timesteps:", timesteps)

    # Read the 'data/timestep_0' dataset
    timestep_0 = f["data/timestep_00"][:]
    print("Shape of timestep_00:", timestep_0.shape)

Timesteps: [  1.45807004   3.45807004   5.45807004   7.45807004   9.45807004
  11.45807004  13.45807004  15.45807004  17.45807004  19.45807004
  21.38174194  22.99592707  24.50782963  26.19726356  27.99843952
  29.75765043  31.51438702  33.20381582  34.95135723  36.72389513
  38.47481936  40.01190191  41.61454881  43.21199175  45.18455608
  47.12633792  49.10099627  50.85762921  52.84008689  54.82849974
  56.31218136  57.75015469  59.60150863  61.15680753  63.00882087
  65.00882087  67.00882087  69.00882087  71.00882087  73.00882087
  75.00882087  77.00882087  79.00882087  81.00882087  83.00882087
  84.94536014  86.94467699  88.94467699  90.94467699  92.94280683
  94.90464083  96.90464083  98.90464083 100.90464083 102.90464083
 104.90464083 106.90464083 108.90464083 110.90464083 112.90464083
 114.90464083 116.90407441 118.90407441 120.90407441 122.90407441
 124.90407441 126.90407441 128.90407441 130.90407441 132.90407441
 134.90407441 136.90407441 138.90407441 140.90407441 142.86067212